# Guradian of truth (Baseline)

In [1]:
!pip install xgboost

In [2]:
import time
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from dataclasses import dataclass
from sklearn.metrics import average_precision_score

from transformers import AutoModelForCausalLM, AutoTokenizer

import xgboost as xgb

## 1. Загрузка модели

In [ ]:
import gc, os

MODEL_ID = "ai-sage/GigaChat3-10B-A1.8B-bf16"
OFFLOAD_DIR = "/tmp/offload"
os.makedirs(OFFLOAD_DIR, exist_ok=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,               # bfloat16 → float16 (для T4)
    max_memory={0: "14GiB", 1: "14GiB"},# убрать     # распределение памяти между двумя GPU
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
DATA_PATH = '/kaggle/input/datasets/stupidhouse/fact-chd-v2/factchd_ru (22) - factchd_ru (22).csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df = df.dropna()
print(df.shape)
#df = df[:100] #тест
df_train = df.sample(frac=0.7, random_state=42)
df_test = df.drop(df_train.index)
df.head()

In [ ]:
import gc
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("CUDA cache cleared")
else:
    print("CUDA is not available")

## 2. Извлечение фичей

Для каждого примера выполняется один forward pass с хуками `with accumulator:`

Затем на соснове взятых проб считаются статистики которые играют роль фичей `FeatureAccumulator.compute_features()`.

In [ ]:
PROBE_LAYERS = [0, 5, 10, 15, 20, 25]


def _exec_device(module) -> torch.device:
    """Устройство для вычислений модуля — корректно работает с disk/CPU offload (meta-тензоры)."""
    if hasattr(module, "_hf_hook"):
        return module._hf_hook.execution_device
    try:
        p = next(module.parameters())
        if p.device.type != "meta":
            return p.device
    except StopIteration:
        pass
    return torch.device("cpu")


class FeatureAccumulator:
    """Collects raw tensors from hooks and computes final feature vectors."""

    def __init__(self, model, probe_layers: list[int] = PROBE_LAYERS):
        self.model = model
        self.probe_layers = probe_layers
        self._hooks: list[torch.utils.hooks.RemovableHook] = []
        self._hidden: dict[str, torch.Tensor] = {}

    def attach(self):
        self._hidden.clear()
        for idx in self.probe_layers:
            name = f"layer_{idx}"

            def _make(n):
                def _fn(_, __, out):
                    h = out[0] if isinstance(out, tuple) else out
                    self._hidden[n] = h.detach()#.cpu()
                return _fn

            self._hooks.append(
                self.model.model.layers[idx].register_forward_hook(_make(name))
            )

    def detach(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def __enter__(self):
        self.attach()
        return self

    def __exit__(self, *_):
        self.detach()

    def compute_features(
        self,
        logits: torch.Tensor,
        input_ids: torch.Tensor,
        answer_start: int,
    ) -> dict:
        seq_len = input_ids.shape[1]
        n_answer = seq_len - answer_start

        ####################################
        #       uncertainty features       #
        ####################################
        answer_logits = logits[0, answer_start - 1: seq_len - 1, :].float().cpu()
        answer_ids = input_ids[0, answer_start:seq_len].cpu()
        log_probs = torch.log_softmax(answer_logits, dim=-1)
        token_lp = log_probs.gather(1, answer_ids.unsqueeze(1)).squeeze(-1)

        probs = torch.softmax(answer_logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
        top1 = probs.max(dim=-1).values
        top5 = probs.topk(min(5, probs.shape[-1]), dim=-1).values.sum(dim=-1)

        uncertainty_arr = np.array([
            token_lp.mean().item(),
            token_lp.min().item(),
            token_lp.max().item(),
            token_lp.std().item() if n_answer > 1 else 0.0,
            entropy.mean().item(),
            entropy.max().item(),
            entropy.std().item() if n_answer > 1 else 0.0,
            torch.exp(-token_lp.mean()).item(),
            float(n_answer),
            token_lp[0].item(),
            top1.mean().item(),
            top5.mean().item(),
        ], dtype=np.float32)

        ###################################
        #        internal features        #
        ###################################

        
        hidden_states = [self._hidden[f"layer_{layer}"] for layer in self.probe_layers] #массив скрытых слоев по всем PROBE_LAYERS

        probe_vectors = [hs[0, answer_start - 1].detach().cpu().float().numpy() for hs in hidden_states ] # получили массив векторов

        norm_dev = _exec_device(self.model.model.norm)
        head_dev = _exec_device(self.model.lm_head)

        int_scalars = []
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            int_scalars.append(hs[answer_start - 1].norm().item())
            int_scalars.append(hs[answer_start: seq_len].norm(dim=-1).mean().item())

            ans_hs = hs[answer_start - 1: seq_len - 1].unsqueeze(0)
            with torch.no_grad():
                normed = self.model.model.norm(ans_hs.to(norm_dev))
                ll = self.model.lm_head(normed.to(head_dev)).float().cpu()
            ll_p = torch.softmax(ll[0], dim=-1)
            ll_e = -(ll_p * torch.log(ll_p + 1e-10)).sum(dim=-1)
            int_scalars.append(ll_e.mean().item())

        first_e = int_scalars[2]
        last_e = int_scalars[-1]
        int_scalars.append(first_e - last_e)
        int_scalars.append(last_e / (first_e + 1e-10))

        internal_scalar_arr = np.array(int_scalars, dtype=np.float32)

        self._hidden.clear()
        return {
            "uncertainty": uncertainty_arr,
            "internal_scalars": internal_scalar_arr,
            "probe_vecs": probe_vectors,
            **{f"probe_vec_{layer}": vec for layer, vec in zip(self.probe_layers, probe_vectors)},
        }


def preprocess(
    tokenizer: AutoTokenizer,
    prompt: str,
    answer: str,
) -> tuple[torch.Tensor, int]:
    messages_prompt = [{"role": "user", "content": prompt}]
    prompt_enc = tokenizer.apply_chat_template(
        messages_prompt,
        add_generation_prompt=True,
        tokenize=True,
    )
    prompt_token_ids = prompt_enc["input_ids"]
    answer_start_idx = len(prompt_token_ids)

    messages_full = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]

    full_enc = tokenizer.apply_chat_template(
        messages_full,
        tokenize=True,
    )

    token_ids = full_enc["input_ids"]

    token_ids = torch.tensor([token_ids], dtype=torch.long)
    return token_ids, answer_start_idx

In [ ]:
import os
import joblib
import numpy as np
import torch
from tqdm import tqdm

CHECKPOINT_PATH = "X_y_train_checkpoint4.joblib"
FINAL_PATH = "X_y_train.joblib"
SAVE_EVERY = 50

MAX_ROWS = None

def get_input_device(model) -> torch.device:
    """Устройство, куда подавать input_ids (embed_tokens)."""
    return _exec_device(model.model.embed_tokens)


input_device = get_input_device(model)
print(f"Input device (embed_tokens): {input_device}")

accumulator = FeatureAccumulator(model)

# =========================
# Resume из чекпоинта
# =========================
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = joblib.load(CHECKPOINT_PATH)

    unc_list = checkpoint["unc_list"]
    int_list = checkpoint["int_list"]
    probe_lists = checkpoint["probe_lists"]
    label_list = checkpoint["label_list"]
    start_pos = checkpoint["next_pos"]

    print(f"Найден чекпоинт. Продолжаю с позиции {start_pos}.")
else:
    unc_list, int_list = [], []
    probe_lists = {label: [] for label in PROBE_LAYERS}
    label_list = []
    start_pos = 0
    print("Чекпоинт не найден. Начинаю с нуля.")

# =========================
# Границы прохода
# =========================
end_pos = len(df_train) if MAX_ROWS is None else min(MAX_ROWS, len(df_train))

if start_pos >= end_pos:
    print("Все выбранные примеры уже обработаны по чекпоинту.")
else:
    # =========================
    # Основной цикл
    # =========================
    for pos in tqdm(range(start_pos, end_pos), initial=start_pos, total=end_pos):
        row = df_train.iloc[pos]

        token_ids, answer_start_idx = preprocess(
            tokenizer,
            row["prompt"],
            row["model_answer"]
        )
        token_ids = token_ids.to(input_device)

        with accumulator:
            with torch.no_grad():
                outputs = model(token_ids)

        feats = accumulator.compute_features(
            outputs.logits,
            token_ids,
            answer_start_idx,
        )

        del outputs
        # torch.cuda.empty_cache()

        unc_list.append(feats["uncertainty"])
        int_list.append(feats["internal_scalars"])
  
        for layer in PROBE_LAYERS:
            probe_lists[layer].append(feats[f"probe_vec_{layer}"])
        label_list.append(row["is_hallucination"])

        # =========================
        # Чекпоинт каждые SAVE_EVERY примеров
        # =========================
        processed = pos + 1
        if (processed % SAVE_EVERY == 0) or (processed == end_pos):
            checkpoint = {
                "unc_list": unc_list,
                "int_list": int_list,
                "probe_lists": probe_lists,
                "label_list": label_list,
                "next_pos": processed,   # откуда продолжать
            }
            joblib.dump(checkpoint, CHECKPOINT_PATH)
            print(f"Чекпоинт сохранён: {processed}/{end_pos}")

# =========================
# Сборка финального объекта
# =========================
if len(unc_list) > 0:
    X_y_train = {
        "uncertainty_X":     np.stack(unc_list).astype(np.float32),
        "internal_scalar_X": np.stack(int_list).astype(np.float32),
        **{f"probe_vec_{layer}":  np.stack(probe_lists[layer]).astype(np.float32) for layer in PROBE_LAYERS},
        "labels":            np.array(label_list, dtype=np.int32),
    }

    joblib.dump(X_y_train, FINAL_PATH)
    print(f"Финальный X_y_train сохранён в: {FINAL_PATH}")
    print("Shapes:")
    for k, v in X_y_train.items():
        print(f"  {k}: {v.shape}")
else:
    print("Пока нет данных для сборки X_y_train.")

## 3. Обучение классификатора

In [ ]:
class HallucinationClassifier:
    def __init__(self, probe_key="probe_vec_25"):
        self.probe_key = probe_key
        self.pca = PCA(n_components=64, random_state=42)
        self.scaler = StandardScaler()
        self.clf = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=1.0,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
        self.threshold = 0.5

    def fit(self, X_y: dict):
        """Fit from pre-extracted features (npz)."""
        y = X_y["labels"]

        probe_last_prompt_pca = self.pca.fit_transform(
            X_y[self.probe_key]
        )
        X = np.hstack([
            probe_last_prompt_pca,
            X_y["internal_scalar_X"],
            X_y["uncertainty_X"],
        ])

        X = self.scaler.fit_transform(X)
        # Автоматическая балансировка весов классов
        neg, pos = (y == 0).sum(), (y == 1).sum()
        self.clf.set_params(scale_pos_weight=neg / pos if pos > 0 else 1.0)
        self.clf.fit(X, y)

    def to_X(self, features: dict) -> np.ndarray:
        if features[self.probe_key].ndim == 1:
            return np.hstack([
                self.pca.transform(features[self.probe_key].reshape(1, -1)),
                features["internal_scalars"].reshape(1, -1),
                features["uncertainty"].reshape(1, -1),
            ])
        else:
            return np.hstack([
                self.pca.transform(features[self.probe_key]),
                features["internal_scalars"],
                features["uncertainty"],
            ])

    def predict_proba(self, features: dict) -> float:
        X_s = self.scaler.transform(self.to_X(features))
        proba = self.clf.predict_proba(X_s)[0, 1]   # вероятность класса 1 (галлюцинация)
        return float(proba)

    def predict(self, features: dict) -> tuple[int, float]:
        prob = self.predict_proba(features)
        return int(prob >= self.threshold), prob

In [ ]:
clf = HallucinationClassifier(probe_key="probe_vec_20")

clf.fit(X_y_train)

## 4. Инференс

In [ ]:
@dataclass
class ScoringResult:
    is_hallucination: bool
    is_hallucination_proba: float
    t_model_sec: float = 0.0
    t_overhead_sec: float = 0.0
    t_total_sec: float = 0.0


class GuardianOfTruth:
    def __init__(self, model, tokenizer, classifier: HallucinationClassifier):
        self.model = model
        self.tokenizer = tokenizer
        self.classifier = classifier
        self.accumulator = FeatureAccumulator(model)

    def score(self, prompt: str, answer: str) -> ScoringResult:
        token_ids, answer_start_idx = preprocess(self.tokenizer, prompt, answer)
        device = next(self.model.parameters()).device
        token_ids = token_ids.to(device)

        ############################
        #     Gigachat forward     #
        ############################
        t0 = time.perf_counter()
        with self.accumulator:
            with torch.no_grad():
                outputs = self.model(token_ids)
        t_model = time.perf_counter() - t0

        ########################
        #      Classifier      #
        ########################
        t1 = time.perf_counter()
        features = self.accumulator.compute_features(
            outputs.logits,
            token_ids,
            answer_start_idx,
        )
        del outputs
        torch.cuda.empty_cache()

        if features is None:
            return ScoringResult(is_hallucination=False, is_hallucination_proba=0.0)

        label, prob = self.classifier.predict(features)
        t_overhead = time.perf_counter() - t1

        return ScoringResult(
            is_hallucination=bool(label),
            is_hallucination_proba=prob,
            t_model_sec=t_model,
            t_overhead_sec=t_overhead,
            t_total_sec=t_model + t_overhead,
        )

## 5. Метрики

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import average_precision_score

# =========================
# Настройки
# =========================
CHECKPOINT_DIR = "guard_checkpoints2"
SAVE_EVERY = 1000

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def run_guard_split_with_checkpoint(
    df,
    split_name,
    guard,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every=SAVE_EVERY,
):
    """
    Считает score по датасету с чекпоинтами и resume.
    Сохраняет только нужные поля, а не весь объект res.
    """
    checkpoint_path = os.path.join(checkpoint_dir, f"{split_name}_checkpoint15.joblib")

    # ---------- resume ----------
    if os.path.exists(checkpoint_path):
        checkpoint = joblib.load(checkpoint_path)
        results_rows = checkpoint["results_rows"]
        start_pos = checkpoint["next_pos"]
        print(f"[{split_name}] найден чекпоинт, продолжаю с позиции {start_pos}")
    else:
        results_rows = []
        start_pos = 0
        print(f"[{split_name}] чекпоинт не найден, начинаю с нуля")

    total = len(df)

    try:
        for pos in tqdm(range(start_pos, total), initial=start_pos, total=total, desc=split_name):
            row = df.iloc[pos]
            res = guard.score(row["prompt"], row["model_answer"])

            # сохраняем только сериализуемые значения
            results_rows.append({
                "pred_proba": float(res.is_hallucination_proba),
                "t_model_ms": float(res.t_model_sec * 1000) if hasattr(res, "t_model_sec") else np.nan,
                "t_overhead_ms": float(res.t_overhead_sec * 1000) if hasattr(res, "t_overhead_sec") else np.nan,
            })

            processed = pos + 1

            # чекпоинт каждые save_every примеров
            if (processed % save_every == 0) or (processed == total):
                checkpoint = {
                    "results_rows": results_rows,
                    "next_pos": processed,
                }
                joblib.dump(checkpoint, checkpoint_path)
                print(f"[{split_name}] чекпоинт сохранён: {processed}/{total}")

    except KeyboardInterrupt:
        # сохраняем то, что уже успели посчитать
        checkpoint = {
            "results_rows": results_rows,
            "next_pos": len(results_rows),
        }
        joblib.dump(checkpoint, checkpoint_path)
        print(f"\n[{split_name}] остановлено вручную, прогресс сохранён: {len(results_rows)}/{total}")
        return None

    # ---------- собираем scored df ----------
    df_scored = df.iloc[:len(results_rows)].copy()
    df_scored["pred_proba"] = [r["pred_proba"] for r in results_rows]
    df_scored["t_model_ms"] = [r["t_model_ms"] for r in results_rows]
    df_scored["t_overhead_ms"] = [r["t_overhead_ms"] for r in results_rows]

    return df_scored

In [ ]:
# =========================
# Запуск train
# =========================
guard = GuardianOfTruth(model, tokenizer, clf)

df_train_scored = run_guard_split_with_checkpoint(
    df=df_train,
    split_name="train",
    guard=guard,
)

# =========================
# Запуск test
# =========================
df_test_scored = run_guard_split_with_checkpoint(
    df=df_test,
    split_name="test",
    guard=guard,
)

# =========================
# Метрики считаем только если оба прохода завершены
# =========================
if (df_train_scored is not None) and (df_test_scored is not None):
    metrics_df = pd.DataFrame([
        {
            "split": "Train",
            "pr_auc": round(
                average_precision_score(
                    df_train_scored["is_hallucination"],
                    df_train_scored["pred_proba"].values
                ),
                4
            )
        },
        {
            "split": "Test",
            "pr_auc": round(
                average_precision_score(
                    df_test_scored["is_hallucination"],
                    df_test_scored["pred_proba"].values
                ),
                4
            )
        },
    ]).set_index("split")

    print(metrics_df.to_string())
    print()
    print("Timing (test, per sample):")
    print(f"  model forward : {df_test_scored['t_model_ms'].mean():.1f} ms (mean)")
    print(f"  overhead      : {df_test_scored['t_overhead_ms'].mean():.1f} ms (mean)")
    print(f"  overhead p99  : {df_test_scored['t_overhead_ms'].quantile(0.99):.1f} ms")
    overhead_ok = df_test_scored["t_overhead_ms"].quantile(0.99) < 1000
    print(f"  {'PASS' if overhead_ok else 'FAIL'}: p99 overhead < 1000 ms")
else:
    print("Один из проходов остановлен раньше времени. Запусти ячейку ещё раз — она продолжит с чекпоинта.")